In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Calibration fit: CeO₂, PEARL

Fit the CeO₂ standard to time-of-flight neutron powder data from PEARL at ISIS: the calibration
and the profile, with a polynomial background.

The project is the catalogue case `pd-neut-tof_ceo2-pearl`, the one the command line and the
app's Examples open.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Single</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">tof</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-tof_ceo2-pearl', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['ceo2']
experiment = project.experiments['pearl']

Data 'pd-neut-tof_ceo2-pearl' is at data/pd-neut-tof_ceo2-pearl-d212831a55d0/pd-neut-tof_ceo2-pearl


The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('pearl')

View(html=<382,322 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale and time-of-flight calibration

CeO₂ is the calibration standard, so its cell stays fixed and the DIFC terms put the peaks in
place.

In [6]:
experiment.linked_structures['ceo2'].scale.free = True
experiment.instrument.calib_d_to_tof_offset.free = True
experiment.instrument.calib_d_to_tof_linear.free = True
experiment.instrument.calib_d_to_tof_quadratic.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,pearl,instrument,,calib_d_to_tof_offset,0,,-inf,inf,µs
2,pearl,instrument,,calib_d_to_tof_linear,4.68×10³,,-inf,inf,µs/Å
3,pearl,instrument,,calib_d_to_tof_quadratic,1.7621,,-inf,inf,µs/Å²
4,pearl,linked_structure,ceo2,scale,2.00×10³,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 4 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       14.21        27.74
      1      0.009     13.04        23.36   15.8% ↓
      2      0.017     13.03        23.34   0.1% ↓
      3      0.026     13.03        23.34   
      4      0.040     13.03        23.34   
  done · converged · 4 iters · reduced χ² 23.34 · Rwp 13.03 % · 0.05 s


View(html=<81,463 characters>, height=376)

In [9]:
project.display.pattern('pearl')

View(html=<383,287 characters>, height=480)

### Background

The six polynomial terms follow the level between the peaks.

In [10]:
for term in experiment.background_terms:
    term.coef.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 10 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.03        23.41
      1      0.014      6.73         6.24   73.4% ↓
      2      0.023      6.73         6.24   
      3      0.037      6.73         6.24   
  done · converged · 3 iters · reduced χ² 6.24 · Rwp 6.73 % · 0.05 s


View(html=<87,588 characters>, height=376)

In [12]:
project.display.pattern('pearl')

View(html=<405,004 characters>, height=480)

### Gaussian widths

The Gaussian terms that grow with d spacing.

In [13]:
experiment.peak.broad_gauss_sigma_1.free = True
experiment.peak.broad_gauss_sigma_2.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 12 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.73         6.24
      1      0.012      6.28         5.44   12.9% ↓
      2      0.018      6.27         5.43   0.1% ↓
      3      0.032      6.27         5.43   
  done · converged · 3 iters · reduced χ² 5.43 · Rwp 6.27 % · 0.04 s


View(html=<89,786 characters>, height=376)

In [15]:
project.display.pattern('pearl')

View(html=<404,874 characters>, height=480)

### Lorentzian width

The Lorentzian term that grows with d spacing.

In [16]:
experiment.peak.broad_lorentz_gamma_1.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 13 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.27         5.43
      1      0.013      6.19         5.30   2.5% ↓
      2      0.021      6.19         5.30   
      3      0.034      6.19         5.30   
  done · converged · 3 iters · reduced χ² 5.30 · Rwp 6.19 % · 0.04 s


View(html=<90,884 characters>, height=376)

In [18]:
project.display.pattern('pearl')

View(html=<404,850 characters>, height=480)

### Rise and decay

The back-to-back exponentials that shape each time-of-flight peak.

In [19]:
experiment.peak.rise_alpha_1.free = True
experiment.peak.decay_beta_0.free = True
experiment.peak.decay_beta_1.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 16 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.19         5.30
      1      0.012      6.02         5.01   5.4% ↓
      2      0.018      5.67         4.45   11.3% ↓
      3      0.024      5.67         4.44   0.3% ↓
      4      0.037      5.67         4.44   
  done · converged · 4 iters · reduced χ² 4.44 · Rwp 5.67 % · 0.05 s


View(html=<93,971 characters>, height=376)

In [21]:
project.display.pattern('pearl')

View(html=<404,656 characters>, height=480)

### Displacements

In [22]:
structure.atom_sites['Ce'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 18 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.67         4.44
      1      0.013      5.30         3.88   12.5% ↓
      2      0.019      5.29         3.88   0.2% ↓
      3      0.028      5.29         3.88   
      4      0.042      5.29         3.88   
  done · converged · 4 iters · reduced χ² 3.88 · Rwp 5.29 % · 0.05 s


View(html=<96,059 characters>, height=376)

In [24]:
project.display.pattern('pearl')

View(html=<404,887 characters>, height=480)

### Absorption

Absorption weakens the intensities with time of flight, as the displacements do, so it comes
after them, on its own.

In [25]:
experiment.absorption.abscor1.free = True

In [26]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_ceo2-pearl, 2237 pts fitted / 2524 loaded, 19 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.29         3.88
      1      0.018      5.29         3.88   
      2      0.034      5.29         3.88   
  done · converged · 2 iters · reduced χ² 3.88 · Rwp 5.29 % · 0.04 s


View(html=<97,029 characters>, height=376)

In [27]:
project.display.pattern('pearl')

View(html=<404,906 characters>, height=480)

In [28]:
project.display.fit.correlations()

,1,2,3,4,5
1 pearl.linked_structure.ceo2.scale,,,,,
2 pearl.background.3.coef,-0.04,,,,
3 pearl.background.5.coef,0.19,-0.92,,,
4 pearl.background.6.coef,-0.14,0.89,-0.95,,
5 pearl.absorption.abscor1,0.96,-0.05,0.18,-0.14,


In [29]:
project.display.structure('ceo2')

View(html=<203,700 characters>, height=660)

## Save the project

In [30]:
project.save_as('pd-neut-tof_ceo2-pearl')
project.report.save_html('pd-neut-tof_ceo2-pearl.html')

Saving project to 'pd-neut-tof_ceo2-pearl'
Saving report to 'pd-neut-tof_ceo2-pearl.html'
